# Agentic causal-frame policy–sentiment gap analysis

This pipeline is independent of the semantic-coverage and causal-frame-network
notebooks. It reads only:

```text
causal_nlp/output/shared/clean_sentence_inventory.csv
```

The workflow:

1. selects sentences containing explicit English or French causal cues;
2. asks DeepSeek to compare cause–relation–effect frames across policy and sentiment;
3. independently verifies the proposed findings with reordered evidence;
4. matches findings across repeated runs using frame-level similarity;
5. retains only findings supported across the required number of runs;
6. preserves human labels and calculates precision, recall and \(F_1\).

The pipeline does not read `output/frame_network` or `output/semantic_coverage`.


In [ ]:
from __future__ import annotations

import difflib
import itertools
import json
import math
import os
import random
import re
import sys
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 240)

PIPELINE_VERSION = "frame-agent-v2"

# Workload
RANDOM_STATE = int(os.environ.get("AGENT_RANDOM_STATE", "42"))
ANALYSIS_RUNS = max(2, int(os.environ.get("AGENT_ANALYSIS_RUNS", "2")))
BATCHES_PER_SCOPE = max(1, int(os.environ.get("AGENT_BATCHES_PER_SCOPE", "2")))
SENTENCES_PER_CORPUS = max(
    6, int(os.environ.get("AGENT_SENTENCES_PER_CORPUS", "24"))
)
MAX_FINDINGS = max(1, int(os.environ.get("AGENT_MAX_FINDINGS", "6")))
MAX_CASES = max(0, int(os.environ.get("AGENT_MAX_CASES", "0")))

# Scope and quality thresholds
MIN_SCOPE_SENTENCES = max(
    1, int(os.environ.get("AGENT_MIN_SCOPE_SENTENCES", "10"))
)
MIN_SCOPE_SOURCES = max(
    1, int(os.environ.get("AGENT_MIN_SCOPE_SOURCES", "2"))
)
MIN_CONFIDENCE = float(os.environ.get("AGENT_MIN_CONFIDENCE", "0.65"))
MIN_FAITHFULNESS = float(os.environ.get("AGENT_MIN_FAITHFULNESS", "0.85"))
MIN_FRAME_MATCH = float(os.environ.get("AGENT_MIN_FRAME_MATCH", "0.48"))
REQUIRED_ACCEPTED_RUNS = int(
    os.environ.get("AGENT_REQUIRED_ACCEPTED_RUNS", str(ANALYSIS_RUNS))
)
REQUIRED_ACCEPTED_RUNS = min(
    max(2, REQUIRED_ACCEPTED_RUNS),
    ANALYSIS_RUNS,
)

RUN_AGENT = os.environ.get(
    "RUN_DEEPSEEK_AGENT",
    "1" if os.environ.get("DEEPSEEK_API_KEY") else "0",
).strip().lower() not in {"0", "false", "no", "off"}

RESUME_RUNS = os.environ.get(
    "AGENT_RESUME_RUNS", "1"
).strip().lower() not in {"0", "false", "no", "off"}


def find_causal_nlp_root() -> Path:
    configured = os.environ.get("CAUSAL_NLP_ROOT")
    candidates: list[Path] = []

    if configured:
        candidates.append(Path(configured).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        *cwd.parents,
        cwd / "progress" / "causal_nlp",
    ])

    checked: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.add(candidate)

        if (
            (candidate / "agent.py").exists()
            and (
                candidate
                / "output"
                / "shared"
                / "clean_sentence_inventory.csv"
            ).exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate causal_nlp/agent.py and "
        "causal_nlp/output/shared/clean_sentence_inventory.csv. "
        "Set CAUSAL_NLP_ROOT to the causal_nlp directory."
    )


METHOD_ROOT = find_causal_nlp_root()
SHARED_INPUT = (
    METHOD_ROOT / "output" / "shared" / "clean_sentence_inventory.csv"
)
OUTPUT_DIR = METHOD_ROOT / "output" / "agentic_frame_gap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(METHOD_ROOT) not in sys.path:
    sys.path.insert(0, str(METHOD_ROOT))

from agent import ask_agent

RUNS_PATH = OUTPUT_DIR / "frame_agent_runs.jsonl"
STABILITY_PATH = OUTPUT_DIR / "frame_agent_stability.csv"
CANDIDATES_PATH = OUTPUT_DIR / "frame_agent_candidates.csv"
FINDINGS_PATH = OUTPUT_DIR / "frame_agent_findings.csv"
GAPS_PATH = OUTPUT_DIR / "frame_agent_gaps.csv"
REVIEW_PATH = OUTPUT_DIR / "frame_agent_human_review.csv"
EVALUATION_PATH = OUTPUT_DIR / "frame_agent_evaluation.csv"

RUN_CONFIG = {
    "pipeline_version": PIPELINE_VERSION,
    "analysis_runs": ANALYSIS_RUNS,
    "batches_per_scope": BATCHES_PER_SCOPE,
    "sentences_per_corpus": SENTENCES_PER_CORPUS,
    "maximum_findings": MAX_FINDINGS,
    "minimum_confidence": MIN_CONFIDENCE,
    "minimum_faithfulness": MIN_FAITHFULNESS,
    "minimum_frame_match": MIN_FRAME_MATCH,
    "required_accepted_runs": REQUIRED_ACCEPTED_RUNS,
}

print("Causal-NLP folder:", METHOD_ROOT)
print("Shared input:", SHARED_INPUT)
print("Output folder:", OUTPUT_DIR)
print("DeepSeek enabled:", RUN_AGENT)
print("Analysis runs:", ANALYSIS_RUNS)
print("Required accepted runs:", REQUIRED_ACCEPTED_RUNS)
print("Frame-match threshold:", MIN_FRAME_MATCH)
print("Resume compatible runs:", RESUME_RUNS)

In [ ]:
REQUIRED_COLUMNS = {
    "sentence_id",
    "clean_sentence",
    "corpus",
    "source_type",
    "source_file",
    "doc_id",
    "chunk_id",
    "country",
    "heading_context",
    "synthetic_type",
}

inventory = pd.read_csv(SHARED_INPUT)
missing = REQUIRED_COLUMNS.difference(inventory.columns)
if missing:
    raise ValueError(f"Shared inventory is missing columns: {sorted(missing)}")

for column in REQUIRED_COLUMNS:
    inventory[column] = inventory[column].fillna("").astype(str)

if inventory["sentence_id"].duplicated().any():
    examples = inventory.loc[
        inventory["sentence_id"].duplicated(keep=False),
        "sentence_id",
    ].head(10).tolist()
    raise ValueError(f"sentence_id must be unique. Examples: {examples}")

synthetic_mask = (
    inventory["source_type"].str.strip().str.lower().eq("synthetic")
    | inventory["synthetic_type"].str.strip().ne("")
)

empirical = inventory[
    ~synthetic_mask
    & inventory["corpus"].str.strip().str.lower().isin(
        ["policy", "sentiment"]
    )
].copy()

empirical["corpus"] = empirical["corpus"].str.strip().str.lower()
empirical["clean_sentence"] = empirical["clean_sentence"].str.strip()
empirical = empirical[empirical["clean_sentence"].ne("")].copy()


def infer_country(row: pd.Series) -> str:
    current = str(row.get("country", "")).strip().lower()
    if current and current not in {
        "nan", "none", "other", "unknown", "global"
    }:
        return current.replace(" ", "_")

    source = " ".join(
        str(row.get(column, ""))
        for column in ["source_file", "doc_id", "heading_context"]
    ).lower()

    country_tokens = {
        "ireland": ["ireland", "irish", "qqi"],
        "france": ["france", "french", "français", "francais", "ifop"],
        "australia": ["australia", "australian"],
        "united_states": [
            "united states", "usa", "u.s.", "american"
        ],
    }

    for country, tokens in country_tokens.items():
        if any(token in source for token in tokens):
            return country
    return "other"


empirical["analysis_country"] = empirical.apply(infer_country, axis=1)

if empirical.empty:
    raise ValueError("No empirical policy or sentiment sentences were found.")

support = empirical.groupby("corpus", as_index=False).agg(
    sentences=("sentence_id", "count"),
    sources=("source_file", "nunique"),
)
display(support)

In [ ]:
# Explicit causal cues. The agent receives no arbitrary fallback sentences.
STRICT_CAUSAL_CUE = re.compile(
    r"""
    \b(?:
        because|therefore|thus|hence|consequently|as\sa\sresult|
        due\sto|owing\sto|results?\sin|resulting\sin|
        leads?\sto|leading\sto|causes?|caused\sby|
        depends?\son|dependent\son|requires?|required\sfor|
        necessary\sfor|enables?|enabled\sby|
        prevents?|prevented\sby|reduces?|reduced\sby|
        increases?|increased\sby|exacerbates?|mitigates?|
        poses?\s(?:a\s)?risk|creates?\s(?:a\s)?risk|
        threatens?|protects?\sagainst|in\sorder\sto|so\sthat|
        afin\sde|parce\sque|car|donc|ainsi|par\scons[ée]quent|
        en\sraison\sde|entra[iî]ne|entra[iî]nant|
        conduit\s[àa]|provoque|r[ée]sulte\sde|
        d[ée]pend\sde|d[ée]pendre\sde|n[ée]cessite|
        est\sn[ée]cessaire\spour|permet\sde|permettent\sde|
        emp[êe]che|pr[ée]vient|r[ée]duit|augmente|
        aggrave|att[ée]nue|constitue\sun\srisque|
        menace|prot[èe]ge\scontre
    )\b
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

MODAL_CAUSAL_CUE = re.compile(
    r"""
    \b(?:
        may|might|can|could|is\slikely\sto|is\sexpected\sto|
        has\sthe\spotential\sto|helps?\sto|supports?\s(?:the\s)?|
        risks?\s(?:causing|creating|reducing|increasing)|
        needs?\sto|must|should|
        peut|pourrait|risque\sde|devrait|doit|aide\s[àa]|
        favorise|soutient
    )\b
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

empirical["strict_causal_cue"] = empirical["clean_sentence"].str.contains(
    STRICT_CAUSAL_CUE, na=False
)
empirical["modal_causal_cue"] = empirical["clean_sentence"].str.contains(
    MODAL_CAUSAL_CUE, na=False
)

empirical["causal_score"] = (
    2 * empirical["strict_causal_cue"].astype(int)
    + empirical["modal_causal_cue"].astype(int)
)

causal_empirical = empirical[
    empirical["causal_score"].gt(0)
].copy().reset_index(drop=True)

if causal_empirical.empty:
    raise ValueError(
        "No explicit causal-cue sentences were found in the shared inventory."
    )

candidate_support = causal_empirical.groupby(
    "corpus", as_index=False
).agg(
    candidate_sentences=("sentence_id", "count"),
    candidate_sources=("source_file", "nunique"),
    strict_sentences=("strict_causal_cue", "sum"),
)
display(candidate_support)

country_candidate_support = causal_empirical.groupby(
    ["analysis_country", "corpus"], as_index=False
).agg(
    candidate_sentences=("sentence_id", "count"),
    candidate_sources=("source_file", "nunique"),
)
display(country_candidate_support)

In [ ]:
ALLOWED_CLASSIFICATIONS = {
    "policy_emphasis",
    "sentiment_emphasis",
    "structural_difference",
    "alignment",
    "insufficient_evidence",
}

SUBSTANTIVE_CLASSIFICATIONS = {
    "policy_emphasis",
    "sentiment_emphasis",
    "structural_difference",
}

ALLOWED_RELATIONS = {
    "cause_increase",
    "reduce_prevent",
    "enable_support",
    "require_depend",
    "risk_threat",
    "expected_improvement",
}

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "because", "by",
    "de", "des", "du", "en", "et", "for", "from", "in", "is",
    "la", "le", "les", "of", "on", "or", "pour", "that", "the",
    "to", "un", "une", "with", "ai", "artificial", "intelligence",
}


def evidence_item(row: pd.Series) -> dict[str, Any]:
    return {
        "evidence_id": str(row["sentence_id"]),
        "corpus": str(row["corpus"]),
        "text": str(row["clean_sentence"]),
        "source_file": str(row["source_file"]),
        "doc_id": str(row["doc_id"]),
        "chunk_id": str(row["chunk_id"]),
        "country": str(row["analysis_country"]),
        "heading_context": str(row["heading_context"]),
        "causal_score": int(row["causal_score"]),
        "strict_causal_cue": bool(row["strict_causal_cue"]),
    }


def source_balanced_sample(
    frame: pd.DataFrame,
    limit: int,
    seed: int,
) -> list[dict[str, Any]]:
    """Prioritise explicit cues while sampling sources in round-robin order."""
    if frame.empty or limit <= 0:
        return []

    rng = random.Random(seed)
    grouped: dict[str, list[dict[str, Any]]] = {}

    for source, group in frame.groupby("source_file", sort=True):
        rows = group.to_dict("records")
        for row in rows:
            row["_random_order"] = rng.random()

        rows.sort(
            key=lambda row: (
                int(row.get("causal_score", 0)),
                int(bool(row.get("strict_causal_cue", False))),
                row["_random_order"],
            ),
            reverse=True,
        )
        grouped[str(source)] = rows

    sources = list(grouped)
    rng.shuffle(sources)
    selected: list[dict[str, Any]] = []

    while sources and len(selected) < limit:
        remaining_sources: list[str] = []
        for source in sources:
            rows = grouped[source]
            if rows and len(selected) < limit:
                row = rows.pop(0)
                row.pop("_random_order", None)
                selected.append(evidence_item(pd.Series(row)))
            if rows:
                remaining_sources.append(source)
        sources = remaining_sources

    return selected


def build_scopes(
    frame: pd.DataFrame,
) -> list[tuple[str, pd.DataFrame]]:
    scopes: list[tuple[str, pd.DataFrame]] = [("global", frame)]

    for country in sorted(frame["analysis_country"].unique()):
        if country == "other":
            continue

        subset = frame[frame["analysis_country"].eq(country)].copy()
        counts = subset.groupby("corpus")["sentence_id"].count().to_dict()
        sources = subset.groupby("corpus")["source_file"].nunique().to_dict()

        enough_sentences = all(
            counts.get(corpus, 0) >= MIN_SCOPE_SENTENCES
            for corpus in ["policy", "sentiment"]
        )
        enough_sources = all(
            sources.get(corpus, 0) >= MIN_SCOPE_SOURCES
            for corpus in ["policy", "sentiment"]
        )

        if enough_sentences and enough_sources:
            scopes.append((country, subset))

    return scopes


def build_packages(
    frame: pd.DataFrame,
) -> list[dict[str, Any]]:
    packages: list[dict[str, Any]] = []

    for scope_index, (scope, subset) in enumerate(build_scopes(frame)):
        for batch_index in range(BATCHES_PER_SCOPE):
            seed = RANDOM_STATE + scope_index * 1000 + batch_index * 100

            policy = source_balanced_sample(
                subset[subset["corpus"].eq("policy")],
                SENTENCES_PER_CORPUS,
                seed,
            )
            sentiment = source_balanced_sample(
                subset[subset["corpus"].eq("sentiment")],
                SENTENCES_PER_CORPUS,
                seed + 1,
            )

            if not policy or not sentiment:
                continue

            evidence = policy + sentiment
            random.Random(seed + 2).shuffle(evidence)

            packages.append({
                "analysis_id": f"{scope}__batch_{batch_index + 1:02d}",
                "scope": scope,
                "batch_id": batch_index + 1,
                "maximum_findings": MAX_FINDINGS,
                "evidence_counts": {
                    "policy": len(policy),
                    "sentiment": len(sentiment),
                    "policy_sources": len(
                        {item["source_file"] for item in policy}
                    ),
                    "sentiment_sources": len(
                        {item["source_file"] for item in sentiment}
                    ),
                    "strict_policy": sum(
                        item["strict_causal_cue"] for item in policy
                    ),
                    "strict_sentiment": sum(
                        item["strict_causal_cue"] for item in sentiment
                    ),
                },
                "evidence": evidence,
            })

    if MAX_CASES > 0:
        packages = packages[:MAX_CASES]
    return packages


def shuffled_package(
    package: dict[str, Any],
    seed: int,
) -> dict[str, Any]:
    value = json.loads(json.dumps(package, ensure_ascii=False))
    random.Random(seed).shuffle(value["evidence"])
    return value


def valid_ids(
    package: dict[str, Any],
    corpus: str | None = None,
) -> set[str]:
    return {
        item["evidence_id"]
        for item in package["evidence"]
        if corpus is None or item["corpus"] == corpus
    }


def parse_bool(value: Any) -> bool | None:
    if value is None or (
        isinstance(value, float) and math.isnan(value)
    ):
        return None

    text = str(value).strip().lower()
    if text in {"1", "true", "yes", "y"}:
        return True
    if text in {"0", "false", "no", "n"}:
        return False
    return None


def normalise_ids(value: Any) -> list[str]:
    if not isinstance(value, list):
        return []
    return list(dict.fromkeys(
        str(item).strip()
        for item in value
        if str(item).strip()
    ))


def to_probability(value: Any) -> float:
    try:
        return min(1.0, max(0.0, float(value)))
    except Exception:
        return 0.0


def normalise_findings(
    value: Any,
) -> list[dict[str, Any]]:
    if not isinstance(value, list):
        return []

    output: list[dict[str, Any]] = []
    for position, item in enumerate(value, start=1):
        if not isinstance(item, dict):
            continue

        record = dict(item)
        record["finding_id"] = str(
            record.get("finding_id") or f"F{position}"
        ).strip()

        record["classification"] = str(
            record.get("classification", "insufficient_evidence")
        ).strip().lower()
        if record["classification"] not in ALLOWED_CLASSIFICATIONS:
            record["classification"] = "insufficient_evidence"

        record["cause_theme"] = str(
            record.get("cause_theme", "")
        ).strip()
        record["relation_family"] = str(
            record.get("relation_family", "")
        ).strip().lower()
        record["effect_theme"] = str(
            record.get("effect_theme", "")
        ).strip()
        record["policy_frame_summary"] = str(
            record.get("policy_frame_summary", "")
        ).strip()
        record["sentiment_frame_summary"] = str(
            record.get("sentiment_frame_summary", "")
        ).strip()
        record["explanation"] = str(
            record.get("explanation", "")
        ).strip()

        for key in [
            "policy_evidence_ids",
            "sentiment_evidence_ids",
            "counterevidence_ids",
        ]:
            record[key] = normalise_ids(record.get(key, []))

        record["policy_relation_explicit"] = (
            parse_bool(record.get("policy_relation_explicit")) is True
        )
        record["sentiment_relation_explicit"] = (
            parse_bool(record.get("sentiment_relation_explicit")) is True
        )
        record["orientation_consistent"] = (
            parse_bool(record.get("orientation_consistent")) is True
        )
        record["confidence"] = to_probability(
            record.get("confidence", 0.0)
        )

        output.append(record)

    return output


def normalised_tokens(value: Any) -> set[str]:
    text = (
        str(value)
        .lower()
        .replace("’", "'")
    )
    tokens = re.findall(r"[a-zà-ÿ0-9]+", text)
    return {
        token for token in tokens
        if token not in STOPWORDS and len(token) > 1
    }


def theme_similarity(left: Any, right: Any) -> float:
    left_text = re.sub(r"\s+", " ", str(left).strip().lower())
    right_text = re.sub(r"\s+", " ", str(right).strip().lower())

    if not left_text or not right_text:
        return 0.0
    if left_text == right_text:
        return 1.0

    left_tokens = normalised_tokens(left_text)
    right_tokens = normalised_tokens(right_text)
    union = left_tokens | right_tokens
    token_jaccard = (
        len(left_tokens & right_tokens) / len(union)
        if union else 0.0
    )
    sequence = difflib.SequenceMatcher(
        None, left_text, right_text
    ).ratio()
    return max(token_jaccard, sequence)


def evidence_similarity(
    left: dict[str, Any],
    right: dict[str, Any],
) -> float:
    left_ids = set(
        left.get("policy_evidence_ids", [])
        + left.get("sentiment_evidence_ids", [])
    )
    right_ids = set(
        right.get("policy_evidence_ids", [])
        + right.get("sentiment_evidence_ids", [])
    )
    union = left_ids | right_ids
    return (
        len(left_ids & right_ids) / len(union)
        if union else 0.0
    )


def frame_similarity(
    left: dict[str, Any],
    right: dict[str, Any],
) -> float:
    """Similarity used only after classification and relation-family agreement."""
    if left.get("classification") != right.get("classification"):
        return 0.0
    if left.get("relation_family") != right.get("relation_family"):
        return 0.0

    cause = theme_similarity(
        left.get("cause_theme"),
        right.get("cause_theme"),
    )
    effect = theme_similarity(
        left.get("effect_theme"),
        right.get("effect_theme"),
    )
    evidence = evidence_similarity(left, right)

    return 0.40 * cause + 0.40 * effect + 0.20 * evidence


def pairwise_frame_jaccard(
    left: list[dict[str, Any]],
    right: list[dict[str, Any]],
) -> float:
    if not left and not right:
        return 1.0
    if not left or not right:
        return 0.0

    possible: list[tuple[float, int, int]] = []
    for left_index, left_item in enumerate(left):
        for right_index, right_item in enumerate(right):
            score = frame_similarity(left_item, right_item)
            if score >= MIN_FRAME_MATCH:
                possible.append((score, left_index, right_index))

    possible.sort(reverse=True)
    used_left: set[int] = set()
    used_right: set[int] = set()
    matched = 0

    for score, left_index, right_index in possible:
        if left_index in used_left or right_index in used_right:
            continue
        used_left.add(left_index)
        used_right.add(right_index)
        matched += 1

    union = len(left) + len(right) - matched
    return matched / union if union else 1.0


def json_cell(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False)


packages = build_packages(causal_empirical)
print("Analysis packages:", len(packages))

if packages:
    display(pd.DataFrame([
        {
            "analysis_id": package["analysis_id"],
            "scope": package["scope"],
            **package["evidence_counts"],
        }
        for package in packages
    ]))

In [ ]:
FRAME_ANALYSIS_PROMPT = r"""
You compare explicit causal frames in cleaned policy and sentiment sentences.
Use only the supplied evidence.

A causal frame contains:
- a cause, prerequisite or enabling factor;
- one relation family;
- an effect, dependent outcome or risk.

Allowed relation families:
cause_increase, reduce_prevent, enable_support, require_depend,
risk_threat, expected_improvement.

Strict evidence rules:
- Use an evidence sentence only when it explicitly expresses the relation.
- Do not convert a topic mention, question, recommendation or general concern
  into a causal relation.
- Do not use phrases such as "indirectly acknowledges", "implies" or "suggests"
  to create a missing relation.
- Do not infer real-world causality.
- Do not treat absence from this evidence batch as corpus-wide absence.
- Every alignment or gap finding must cite explicit policy and sentiment
  relations.
- For policy_emphasis or sentiment_emphasis, the other corpus must contain a
  related explicit frame that provides the contrast. Otherwise return
  insufficient_evidence.
- Alignment requires compatible cause-to-effect orientation.
- Counterevidence must be reported.
- Use compact cause and effect themes of two to six words, preferably using
  words present in the evidence.
- Return at most the requested maximum number of findings.

Classifications:
- policy_emphasis: both corpora express related frames, but policy gives the
  relation materially stronger or more specific emphasis.
- sentiment_emphasis: both corpora express related frames, but sentiment gives
  the relation materially stronger or more specific emphasis.
- structural_difference: both corpora express related causal content but differ
  materially in orientation, relation or outcome.
- alignment: both corpora express a corresponding causal frame.
- insufficient_evidence: the supplied evidence cannot support a reliable
  cross-corpus comparison.

Return one JSON object with exactly this structure:
{
  "analysis_id": "exact input analysis_id",
  "findings": [
    {
      "finding_id": "F1, F2, ...",
      "classification": "policy_emphasis | sentiment_emphasis | structural_difference | alignment | insufficient_evidence",
      "cause_theme": "two to six words",
      "relation_family": "cause_increase | reduce_prevent | enable_support | require_depend | risk_threat | expected_improvement",
      "effect_theme": "two to six words",
      "policy_frame_summary": "explicit policy frame in one sentence",
      "sentiment_frame_summary": "explicit sentiment frame in one sentence",
      "explanation": "no more than 90 words",
      "policy_evidence_ids": ["exact policy IDs"],
      "sentiment_evidence_ids": ["exact sentiment IDs"],
      "counterevidence_ids": ["exact IDs"],
      "confidence": 0.0
    }
  ],
  "limitations": ["short limitations"]
}
""".strip()


FRAME_VERIFICATION_PROMPT = r"""
Independently verify the candidate causal-frame findings using only the
reordered evidence.

For each retained finding:
- check every evidence ID;
- confirm that both corpora explicitly express a causal, enabling, dependency,
  prevention, risk or expected-improvement relation;
- reject indirect topic matches and unexpressed causal links;
- verify cause-to-effect orientation and relation family;
- check the classification and omitted counterevidence;
- avoid treating batch-level absence as corpus-wide absence;
- rewrite the finding when a correct revision is possible.

Return one JSON object with exactly this structure:
{
  "analysis_id": "exact input analysis_id",
  "verdict": "accept | revise | reject",
  "verified_findings": [
    {
      "finding_id": "F1, F2, ...",
      "classification": "policy_emphasis | sentiment_emphasis | structural_difference | alignment | insufficient_evidence",
      "cause_theme": "two to six words",
      "relation_family": "cause_increase | reduce_prevent | enable_support | require_depend | risk_threat | expected_improvement",
      "effect_theme": "two to six words",
      "policy_frame_summary": "explicit policy frame in one sentence",
      "sentiment_frame_summary": "explicit sentiment frame in one sentence",
      "explanation": "no more than 90 words",
      "policy_evidence_ids": ["exact policy IDs"],
      "sentiment_evidence_ids": ["exact sentiment IDs"],
      "counterevidence_ids": ["exact IDs"],
      "policy_relation_explicit": true,
      "sentiment_relation_explicit": true,
      "orientation_consistent": true,
      "confidence": 0.0
    }
  ],
  "invalid_evidence_ids": ["IDs"],
  "unresolved_counterevidence_ids": ["IDs"],
  "evidence_faithfulness": 0.0,
  "reason": "no more than 70 words"
}

Use an empty verified_findings list when no finding survives verification.
""".strip()


ANALYSIS_KEYS = {"analysis_id", "findings", "limitations"}
VERIFICATION_KEYS = {
    "analysis_id",
    "verdict",
    "verified_findings",
    "invalid_evidence_ids",
    "unresolved_counterevidence_ids",
    "evidence_faithfulness",
    "reason",
}

In [ ]:
def run_key(
    analysis_id: str,
    run_index: int,
) -> tuple[str, int]:
    return analysis_id, run_index


def load_previous_runs(
) -> dict[tuple[str, int], dict[str, Any]]:
    previous: dict[tuple[str, int], dict[str, Any]] = {}
    if not RESUME_RUNS or not RUNS_PATH.exists():
        return previous

    with RUNS_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue

            try:
                row = json.loads(line)
            except Exception:
                continue

            if row.get("pipeline_version") != PIPELINE_VERSION:
                continue
            if row.get("run_config") != RUN_CONFIG:
                continue

            try:
                key = run_key(
                    str(row["analysis_id"]),
                    int(row["run_index"]),
                )
            except Exception:
                continue

            previous[key] = row

    return previous


def save_runs(
    rows: Iterable[dict[str, Any]],
) -> None:
    ordered = sorted(
        rows,
        key=lambda row: (
            str(row.get("analysis_id", "")),
            int(row.get("run_index", 0)),
        ),
    )

    with RUNS_PATH.open("w", encoding="utf-8") as handle:
        for row in ordered:
            handle.write(
                json.dumps(row, ensure_ascii=False, default=str)
                + "\n"
            )


def validate_verified_findings(
    package: dict[str, Any],
    verification: dict[str, Any],
) -> tuple[list[dict[str, Any]], list[str]]:
    findings = normalise_findings(
        verification.get("verified_findings", [])
    )

    all_ids = valid_ids(package)
    policy_ids = valid_ids(package, "policy")
    sentiment_ids = valid_ids(package, "sentiment")

    valid_findings: list[dict[str, Any]] = []
    errors: list[str] = []

    for finding in findings:
        finding_id = finding["finding_id"]
        classification = finding["classification"]
        relation_family = finding["relation_family"]

        cited_policy = set(finding["policy_evidence_ids"])
        cited_sentiment = set(finding["sentiment_evidence_ids"])
        cited_counter = set(finding["counterevidence_ids"])

        if relation_family not in ALLOWED_RELATIONS:
            errors.append(
                f"{finding_id}: invalid relation family"
            )
            continue

        if not (
            cited_policy.issubset(policy_ids)
            and cited_sentiment.issubset(sentiment_ids)
            and cited_counter.issubset(all_ids)
        ):
            errors.append(
                f"{finding_id}: invalid or cross-corpus evidence ID"
            )
            continue

        if classification != "insufficient_evidence":
            if not cited_policy or not cited_sentiment:
                errors.append(
                    f"{finding_id}: comparison lacks evidence from both corpora"
                )
                continue

            if not (
                finding["policy_relation_explicit"]
                and finding["sentiment_relation_explicit"]
            ):
                errors.append(
                    f"{finding_id}: one or both relations are not explicit"
                )
                continue

            if not finding["orientation_consistent"]:
                errors.append(
                    f"{finding_id}: cause-effect orientation is inconsistent"
                )
                continue

        if finding["confidence"] < MIN_CONFIDENCE:
            errors.append(
                f"{finding_id}: confidence below threshold"
            )
            continue

        if not finding["explanation"]:
            errors.append(
                f"{finding_id}: missing explanation"
            )
            continue

        finding["machine_valid"] = True
        valid_findings.append(finding)

    return valid_findings, errors


previous_runs = load_previous_runs()
run_rows: list[dict[str, Any]] = []

for package_index, package in enumerate(packages):
    analysis_id = package["analysis_id"]

    for zero_based_run in range(ANALYSIS_RUNS):
        run_index = zero_based_run + 1
        key = run_key(analysis_id, run_index)

        if key in previous_runs:
            print(f"Reusing {analysis_id}, run {run_index}")
            run_rows.append(previous_runs[key])
            continue

        if RUN_AGENT:
            candidate = ask_agent(
                FRAME_ANALYSIS_PROMPT,
                shuffled_package(
                    package,
                    RANDOM_STATE
                    + package_index * 100
                    + zero_based_run,
                ),
                action=f"Analysing {analysis_id}, run {run_index}",
                required_keys=ANALYSIS_KEYS,
                temperature=0.0,
            )

            verification = ask_agent(
                FRAME_VERIFICATION_PROMPT,
                {
                    "analysis_id": analysis_id,
                    "candidate": candidate,
                    "case": shuffled_package(
                        package,
                        RANDOM_STATE
                        + 10000
                        + package_index * 100
                        + zero_based_run,
                    ),
                },
                action=f"Verifying {analysis_id}, run {run_index}",
                required_keys=VERIFICATION_KEYS,
                temperature=0.0,
            )
        else:
            candidate = {
                "analysis_id": analysis_id,
                "findings": [],
                "limitations": ["DeepSeek execution is disabled."],
                "status": "not_run",
            }
            verification = {
                "analysis_id": analysis_id,
                "verdict": "reject",
                "verified_findings": [],
                "invalid_evidence_ids": [],
                "unresolved_counterevidence_ids": [],
                "evidence_faithfulness": 0.0,
                "reason": (
                    "Set DEEPSEEK_API_KEY and "
                    "RUN_DEEPSEEK_AGENT=1."
                ),
                "status": "not_run",
            }

        valid_findings, validation_errors = (
            validate_verified_findings(package, verification)
        )

        faithfulness = to_probability(
            verification.get("evidence_faithfulness", 0.0)
        )
        verdict = str(
            verification.get("verdict", "reject")
        ).strip().lower()

        invalid_evidence_ids = normalise_ids(
            verification.get("invalid_evidence_ids", [])
        )
        unresolved_counterevidence_ids = normalise_ids(
            verification.get(
                "unresolved_counterevidence_ids", []
            )
        )

        analysis_id_matches = (
            str(candidate.get("analysis_id", "")) == analysis_id
            and str(verification.get("analysis_id", ""))
            == analysis_id
        )

        # Invalid individual findings are filtered rather than rejecting
        # every valid finding in the same run.
        machine_accepted = bool(
            analysis_id_matches
            and verdict in {"accept", "revise"}
            and faithfulness >= MIN_FAITHFULNESS
            and not invalid_evidence_ids
            and not unresolved_counterevidence_ids
            and valid_findings
        )

        row = {
            "pipeline_version": PIPELINE_VERSION,
            "run_config": RUN_CONFIG,
            "analysis_id": analysis_id,
            "scope": package["scope"],
            "batch_id": package["batch_id"],
            "run_index": run_index,
            "candidate": candidate,
            "verification": verification,
            "findings": valid_findings,
            "validation_errors": validation_errors,
            "evidence_faithfulness": faithfulness,
            "analysis_id_matches": analysis_id_matches,
            "invalid_evidence_ids": invalid_evidence_ids,
            "unresolved_counterevidence_ids": (
                unresolved_counterevidence_ids
            ),
            "verdict": verdict,
            "machine_accepted": machine_accepted,
            "analysis_model": (
                candidate.get("_agent", {}).get("model")
            ),
            "verification_model": (
                verification.get("_agent", {}).get("model")
            ),
        }

        run_rows.append(row)
        previous_runs[key] = row
        save_runs(previous_runs.values())

print("Completed run records:", len(run_rows))
print(
    "Accepted run records:",
    sum(row["machine_accepted"] for row in run_rows),
)

In [ ]:
def mean_similarity_to_cluster(
    finding: dict[str, Any],
    cluster: dict[str, Any],
) -> float:
    scores = [
        frame_similarity(finding, member["finding"])
        for member in cluster["members"]
    ]
    return float(np.mean(scores)) if scores else 0.0


def cluster_findings(
    accepted_runs: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Create clusters with at most one finding from each run."""
    clusters: list[dict[str, Any]] = []

    for run in sorted(
        accepted_runs,
        key=lambda row: row["run_index"],
    ):
        run_index = int(run["run_index"])

        for finding in run["findings"]:
            best_cluster: dict[str, Any] | None = None
            best_score = 0.0

            for cluster in clusters:
                if run_index in cluster["run_indices"]:
                    continue

                score = mean_similarity_to_cluster(
                    finding,
                    cluster,
                )
                if score >= MIN_FRAME_MATCH and score > best_score:
                    best_cluster = cluster
                    best_score = score

            member = {
                "run_index": run_index,
                "finding": finding,
                "faithfulness": run["evidence_faithfulness"],
                "match_score": best_score if best_cluster else 1.0,
            }

            if best_cluster is None:
                clusters.append({
                    "members": [member],
                    "run_indices": {run_index},
                })
            else:
                best_cluster["members"].append(member)
                best_cluster["run_indices"].add(run_index)

    return clusters


def choose_representative(
    cluster: dict[str, Any],
) -> dict[str, Any]:
    return max(
        cluster["members"],
        key=lambda member: (
            member["finding"]["confidence"],
            member["faithfulness"],
        ),
    )


def cluster_match_score(
    cluster: dict[str, Any],
) -> float:
    findings = [
        member["finding"]
        for member in cluster["members"]
    ]
    if len(findings) < 2:
        return 0.0

    scores = [
        frame_similarity(left, right)
        for left, right in itertools.combinations(findings, 2)
    ]
    return float(np.mean(scores)) if scores else 0.0


def analysis_frame_jaccard(
    accepted_runs: list[dict[str, Any]],
) -> float:
    if len(accepted_runs) < 2:
        return 0.0

    scores = [
        pairwise_frame_jaccard(
            left["findings"],
            right["findings"],
        )
        for left, right in itertools.combinations(
            accepted_runs, 2
        )
    ]
    return float(np.mean(scores)) if scores else 0.0


def consensus_key(
    analysis_id: str,
    finding: dict[str, Any],
) -> str:
    fields = [
        analysis_id,
        finding.get("classification", ""),
        finding.get("relation_family", ""),
        re.sub(
            r"\s+", " ",
            str(finding.get("cause_theme", "")).strip().lower()
        ),
        re.sub(
            r"\s+", " ",
            str(finding.get("effect_theme", "")).strip().lower()
        ),
    ]
    return " | ".join(fields)


def evidence_text(
    evidence_ids: list[str],
) -> str:
    lookup = inventory.set_index("sentence_id")[
        "clean_sentence"
    ].to_dict()
    return " || ".join(
        f"{evidence_id}: {lookup.get(evidence_id, '[missing]')}"
        for evidence_id in evidence_ids
    )


stability_rows: list[dict[str, Any]] = []
candidate_rows: list[dict[str, Any]] = []

for package in packages:
    analysis_id = package["analysis_id"]
    group = sorted(
        [
            row for row in run_rows
            if row["analysis_id"] == analysis_id
        ],
        key=lambda row: row["run_index"],
    )
    if not group:
        continue

    accepted = [
        row for row in group
        if row["machine_accepted"]
    ]
    frame_jaccard = analysis_frame_jaccard(accepted)
    repeated_runs_available = len(accepted) >= 2
    clusters = cluster_findings(accepted)

    stable_cluster_count = 0
    sorted_clusters = sorted(
        clusters,
        key=lambda cluster: (
            choose_representative(cluster)["finding"][
                "classification"
            ],
            choose_representative(cluster)["finding"][
                "relation_family"
            ],
            choose_representative(cluster)["finding"][
                "cause_theme"
            ].lower(),
            choose_representative(cluster)["finding"][
                "effect_theme"
            ].lower(),
        ),
    )

    for cluster_index, cluster in enumerate(
        sorted_clusters, start=1
    ):
        representative_member = choose_representative(cluster)
        representative = representative_member["finding"]
        support_runs = len(cluster["run_indices"])
        match_score = cluster_match_score(cluster)

        machine_retained = bool(
            support_runs >= REQUIRED_ACCEPTED_RUNS
            and match_score >= MIN_FRAME_MATCH
        )
        if machine_retained:
            stable_cluster_count += 1

        classification = representative["classification"]
        machine_is_gap = bool(
            machine_retained
            and classification in SUBSTANTIVE_CLASSIFICATIONS
        )

        policy_ids = representative["policy_evidence_ids"]
        sentiment_ids = representative["sentiment_evidence_ids"]
        counter_ids = representative["counterevidence_ids"]

        candidate_rows.append({
            "analysis_id": analysis_id,
            "scope": group[0]["scope"],
            "batch_id": group[0]["batch_id"],
            "consensus_id": f"C{cluster_index:02d}",
            "consensus_key": consensus_key(
                analysis_id, representative
            ),
            "representative_finding_id": (
                representative["finding_id"]
            ),
            "classification": classification,
            "cause_theme": representative["cause_theme"],
            "relation_family": (
                representative["relation_family"]
            ),
            "effect_theme": representative["effect_theme"],
            "policy_frame_summary": (
                representative["policy_frame_summary"]
            ),
            "sentiment_frame_summary": (
                representative["sentiment_frame_summary"]
            ),
            "explanation": representative["explanation"],
            "policy_evidence_ids": json_cell(policy_ids),
            "sentiment_evidence_ids": json_cell(sentiment_ids),
            "counterevidence_ids": json_cell(counter_ids),
            "policy_evidence_text": evidence_text(policy_ids),
            "sentiment_evidence_text": evidence_text(sentiment_ids),
            "counterevidence_text": evidence_text(counter_ids),
            "confidence": representative["confidence"],
            "evidence_faithfulness": (
                representative_member["faithfulness"]
            ),
            "support_runs": support_runs,
            "required_support_runs": REQUIRED_ACCEPTED_RUNS,
            "frame_match_score": match_score,
            "analysis_frame_jaccard": frame_jaccard,
            "machine_retained": machine_retained,
            "machine_is_gap": machine_is_gap,
        })

    stability_rows.append({
        "analysis_id": analysis_id,
        "scope": group[0]["scope"],
        "batch_id": group[0]["batch_id"],
        "accepted_runs": len(accepted),
        "required_accepted_runs": REQUIRED_ACCEPTED_RUNS,
        "total_runs": len(group),
        "repeated_runs_available": repeated_runs_available,
        "frame_jaccard_stability": frame_jaccard,
        "candidate_clusters": len(clusters),
        "stable_clusters": stable_cluster_count,
        "mean_evidence_faithfulness": (
            float(np.mean([
                row["evidence_faithfulness"]
                for row in accepted
            ]))
            if accepted else 0.0
        ),
    })

stability_df = pd.DataFrame(stability_rows)
candidates_df = pd.DataFrame(candidate_rows)

if candidates_df.empty:
    candidates_df = pd.DataFrame(columns=[
        "analysis_id",
        "scope",
        "batch_id",
        "consensus_id",
        "consensus_key",
        "representative_finding_id",
        "classification",
        "cause_theme",
        "relation_family",
        "effect_theme",
        "policy_frame_summary",
        "sentiment_frame_summary",
        "explanation",
        "policy_evidence_ids",
        "sentiment_evidence_ids",
        "counterevidence_ids",
        "policy_evidence_text",
        "sentiment_evidence_text",
        "counterevidence_text",
        "confidence",
        "evidence_faithfulness",
        "support_runs",
        "required_support_runs",
        "frame_match_score",
        "analysis_frame_jaccard",
        "machine_retained",
        "machine_is_gap",
    ])

findings_df = candidates_df[
    candidates_df["machine_retained"].eq(True)
].copy()

gaps_df = candidates_df[
    candidates_df["machine_is_gap"].eq(True)
].copy()

stability_df.to_csv(STABILITY_PATH, index=False)
candidates_df.to_csv(CANDIDATES_PATH, index=False)
findings_df.to_csv(FINDINGS_PATH, index=False)
gaps_df.to_csv(GAPS_PATH, index=False)

print("Stability rows:", len(stability_df))
print("Consensus candidates:", len(candidates_df))
print("Retained findings:", len(findings_df))
print("Retained causal-frame gaps:", len(gaps_df))

display(stability_df)
display(findings_df.head(20))
display(gaps_df.head(20))

In [ ]:
HUMAN_COLUMNS = [
    "human_is_gap",
    "human_category",
    "human_evidence_faithful",
    "human_confirmed",
    "human_notes",
]


def add_previous_consensus_key(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    value = frame.copy()

    if "consensus_key" in value.columns:
        return value

    needed = {
        "analysis_id",
        "classification",
        "relation_family",
        "cause_theme",
        "effect_theme",
    }
    if needed.issubset(value.columns):
        value["consensus_key"] = value.apply(
            lambda row: consensus_key(
                str(row["analysis_id"]),
                row.to_dict(),
            ),
            axis=1,
        )
    return value


def build_review_table(
    candidates: pd.DataFrame,
) -> pd.DataFrame:
    review = candidates.copy()

    # Human review covers retained and non-retained candidate clusters.
    # This permits measurement of false negatives.
    if REVIEW_PATH.exists():
        previous = pd.read_csv(REVIEW_PATH).fillna("")
        previous = add_previous_consensus_key(previous)

        available = [
            column for column in HUMAN_COLUMNS
            if column in previous.columns
        ]

        if "consensus_key" in previous.columns:
            previous_labels = previous[
                ["consensus_key", *available]
            ].drop_duplicates("consensus_key")

            review = review.merge(
                previous_labels,
                on="consensus_key",
                how="left",
            )

    for column in HUMAN_COLUMNS:
        if column not in review.columns:
            review[column] = ""
        else:
            review[column] = review[column].fillna("")

    preferred_order = [
        "analysis_id",
        "scope",
        "batch_id",
        "consensus_id",
        "consensus_key",
        "classification",
        "cause_theme",
        "relation_family",
        "effect_theme",
        "policy_frame_summary",
        "sentiment_frame_summary",
        "explanation",
        "policy_evidence_ids",
        "sentiment_evidence_ids",
        "counterevidence_ids",
        "policy_evidence_text",
        "sentiment_evidence_text",
        "counterevidence_text",
        "confidence",
        "evidence_faithfulness",
        "support_runs",
        "required_support_runs",
        "frame_match_score",
        "analysis_frame_jaccard",
        "machine_retained",
        "machine_is_gap",
        *HUMAN_COLUMNS,
    ]

    return review[
        [
            column for column in preferred_order
            if column in review.columns
        ]
    ]


review_df = build_review_table(candidates_df)
review_df.to_csv(REVIEW_PATH, index=False)

print("Human-review file:", REVIEW_PATH)
print(
    "Fill human_is_gap with yes/no, save the CSV, "
    "then rerun only the next evaluation cell."
)
display(review_df.head(20))

In [ ]:
def evaluate_human_review(
    review_path: Path,
) -> pd.DataFrame:
    if not review_path.exists():
        return pd.DataFrame([{
            "labelled_findings": 0,
            "true_positives": 0,
            "false_positives": 0,
            "false_negatives": 0,
            "true_negatives": 0,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "status": "human review file not found",
        }])

    review = pd.read_csv(review_path).fillna("")

    required = {"human_is_gap", "machine_is_gap"}
    missing = required.difference(review.columns)
    if missing:
        raise ValueError(
            f"Review file is missing columns: {sorted(missing)}"
        )

    review["human_label"] = review["human_is_gap"].map(
        parse_bool
    )
    labelled = review[
        review["human_label"].notna()
    ].copy()

    if labelled.empty:
        return pd.DataFrame([{
            "labelled_findings": 0,
            "true_positives": 0,
            "false_positives": 0,
            "false_negatives": 0,
            "true_negatives": 0,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "status": "enter yes/no labels in human_is_gap",
        }])

    labelled["machine_label"] = labelled[
        "machine_is_gap"
    ].map(parse_bool)
    labelled["machine_label"] = labelled[
        "machine_label"
    ].fillna(False).astype(bool)
    labelled["human_label"] = labelled[
        "human_label"
    ].astype(bool)

    tp = int(
        (
            labelled["machine_label"]
            & labelled["human_label"]
        ).sum()
    )
    fp = int(
        (
            labelled["machine_label"]
            & ~labelled["human_label"]
        ).sum()
    )
    fn = int(
        (
            ~labelled["machine_label"]
            & labelled["human_label"]
        ).sum()
    )
    tn = int(
        (
            ~labelled["machine_label"]
            & ~labelled["human_label"]
        ).sum()
    )

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return pd.DataFrame([{
        "labelled_findings": len(labelled),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "status": "complete",
    }])


evaluation_df = evaluate_human_review(REVIEW_PATH)
evaluation_df.to_csv(EVALUATION_PATH, index=False)

print("Evaluation file:", EVALUATION_PATH)
display(evaluation_df)

## Outputs

The notebook writes:

- `frame_agent_runs.jsonl`: full analysis and verification records;
- `frame_agent_stability.csv`: accepted runs and frame-level Jaccard stability;
- `frame_agent_candidates.csv`: all repeated-run consensus candidates;
- `frame_agent_findings.csv`: stable gaps and alignments;
- `frame_agent_gaps.csv`: stable substantive gaps only;
- `frame_agent_human_review.csv`: review table with evidence text and preserved labels;
- `frame_agent_evaluation.csv`: precision, recall and \(F_1\).

Old run records from the earlier pipeline are ignored because they do not contain
`pipeline_version = frame-agent-v2`.

To force a new run:

```bash
export AGENT_RESUME_RUNS=0
```

After editing `frame_agent_human_review.csv`, rerun only the final evaluation cell.
